In [3]:
import pandas as pd
import duckdb

# Load data
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
tags = pd.read_csv("tags.csv")
links = pd.read_csv("links.csv")

In [4]:
# Quick checks
print(movies.head())
print(ratings.head())
print(tags.head())
print(links.head())

print(movies.shape, ratings.shape, tags.shape, links.shape)

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
   userId  movieId              tag   timestamp
0       2    60756            funny  1445714994
1       2    60756  Highly quotable  1445714996
2       2    60756     will ferre

In [5]:
# Load into DuckDB
con = duckdb.connect()

con.register("movies_df", movies)
con.register("ratings_df", ratings)
con.register("tags_df", tags)
con.register("links_df", links)

con.execute("CREATE OR REPLACE TABLE movies AS SELECT * FROM movies_df")
con.execute("CREATE OR REPLACE TABLE ratings AS SELECT * FROM ratings_df")
con.execute("CREATE OR REPLACE TABLE tags AS SELECT * FROM tags_df")
con.execute("CREATE OR REPLACE TABLE links AS SELECT * FROM links_df")

In [6]:
# Example join: ratings + movies
genre_ratings = con.execute("""
    SELECT
        m.genres,
        AVG(r.rating) AS avg_rating,
        COUNT(*) AS rating_count
    FROM ratings r
    JOIN movies m
      ON r.movieId = m.movieId
    GROUP BY m.genres
    ORDER BY rating_count DESC
    LIMIT 10
""").fetchdf()

print(genre_ratings)

                      genres  avg_rating  rating_count
0                     Comedy    3.197888          7196
1                      Drama    3.688841          6291
2             Comedy|Romance    3.320015          3967
3       Comedy|Drama|Romance    3.550500          3000
4               Comedy|Drama    3.517187          2851
5              Drama|Romance    3.661910          2838
6    Action|Adventure|Sci-Fi    3.550191          2361
7                Crime|Drama    4.000000          2315
8      Action|Crime|Thriller    3.492600          1554
9  Action|Adventure|Thriller    3.365636          1455
